### Cell-cell communication (CCC) on the cell2location-mapped lymph node - LIANA+

Continues **cell2location01a_tutorial.ipynb** (human lymph node, 10x Visium **V1_Human_Lymph_Node**, 34 reference cell types from the lymphoid-organ scRNA atlas) and reuses the niche ligand-receptor (LR) library of **squid03_niche_LigRec_programs.ipynb** (**src/programs/niche_lr.py**, **programs.py**).

**Inputs** (written by the tutorial notebook, **results/lymph_nodes_analysis/cell2location_map/**)

- **sp.h5ad** - Visium counts + cell2location posterior: 

  - **obsm['q05_cell_abundance_w_sf']** (cells of each type per spot) and
  -  34 **layers[<cell type>]** = counts of every gene attributed to that cell type (they sum to **X**)
- **region_labels.csv** - Leiden regions of the cell-abundance space (**region_cluster**)
- **data/lymphoid_organ_sc.h5ad** - scRNA reference (for cell-type-resolved LIANA)

#### **Analyses**

1. Load the inferred cell abundances, name the c2l regions

2. Spatial weights (Gaussian kernel, ~2 Visium rings)
3. **LIANA bivariate** - global Moran's R + local cosine score for every LR pair (consensus resource)
4. Which **c2l cell types** host each local LR pattern (Spearman local score vs. cell-type fraction)
5. **LIANA inflow** - source cell type (c2l proportion) & ligand -> receptor
6. **cell2location-resolved sender -> receiver** scores from the per-cell-type count layers
7. **LIANA rank_aggregate** on the scRNA reference, filtered by c2l spatial co-localisation
8. **MISTy** - spatial dependencies between c2l cell types (juxta / para views)
9. **niche_lr + programs** - region niches (center vs. neighbours), biological programs, c2l sender/receiver attribution, LIANA cross-check
10. Consensus table across methods

Helpers: **src/programs/c2l_liana.py** (new), **src/programs/niche_lr.py**, **src/programs/programs.py**.

**Caveats.** A Visium spot (55 um, 100 um centre spacing) holds ~27 cells here (median c2l estimate) and no spot is dominated by one cell type (median dominant fraction 0.17), so spot-level clustering by "dominant cell type" is not used. Every sender/receiver attribution comes from the reference signatures combined with c2l abundances: it says which cell type *can* express L/R *where*, not that it does. The reference has no fibroblastic reticular cells (FRC); stromal chemokines (CCL19/CCL21) are attributed to the available stromal types (VSMC, Endo).



In [ ]:
import os, sys, time, importlib
from importlib.metadata import version
from pathlib import Path
import warnings
warnings.filterwarnings("ignore", category=FutureWarning)

import numpy as np
import pandas as pd

import scipy.sparse as sps
from scipy.stats import rankdata

import anndata as ad
import scanpy as sc
import squidpy as sq
import liana as li

import matplotlib as mpl
import matplotlib.pyplot as plt
from matplotlib import rcParams
from matplotlib.colors import LinearSegmentedColormap

rcParams['pdf.fonttype']= 42 # enables correct plotting of text for PDFs

colors = ("#16466d", "#ebd246", "#f30707")
cmap_byr = LinearSegmentedColormap.from_list("blue_yellow_red", list(colors))

import seaborn as sns

root0 = Path('/home/flavio/uv/cell_cell_communication')
ROOT_SRC = root0 / 'src'
if str(ROOT_SRC) not in sys.path:
    sys.path.append(str(ROOT_SRC))

from libs.Basic import create_dir, pdwritecsv, pdreadcsv
from libs.matplotlib_figure_style import apply_figure_style

import programs.niche_lr as nl
import programs.programs as pg
import programs.c2l_liana as cl

import kaleido

print({p: version(p) for p in ['liana', 'scanpy', 'squidpy', 'anndata', 'kaleido']})

In [ ]:
root_data    = root0 / 'data'
root_results = root0 / 'results'
root_figure  = root0 / 'figures'

results_folder = root_results / 'lymph_nodes_analysis'
run_name   = results_folder / 'cell2location_map'          # cell2location01a_tutorial output
file_sp    = run_name / 'sp.h5ad'
file_reg   = run_name / 'region_labels.csv'
file_ref   = root_data / 'lymphoid_organ_sc.h5ad'

root_liana = create_dir(results_folder, 'liana')           # tables / h5ad of this notebook
root_fig_liana = create_dir(root_figure, 'c2l_liana')          # figures of this notebook

apply_figure_style()

print(f"file_sp  {file_sp}: {file_sp.exists()}\nfile_reg {file_reg}: {file_reg.exists()}\nfile_ref {file_ref}: {file_ref.exists()}")

### 1. Inferred cells from cell2location

**cl.load_c2l_spots** reads **sp.h5ad** **without** the 34 cell-type layers (~5 GB; they are read lazily, gene by gene, in section 6):
- **var_names** -> gene symbols (LIANA resources use symbols), **var['gene_ids']** keeps Ensembl

- **layers['counts']** = raw counts, **X** = log1p(CP10k)
- **obsm['c2l_abund']** = q05 posterior abundance (*"at least this many cells"*), **obsm['c2l_frac']** = abundance / total

In [ ]:
adata_pos = cl.load_c2l_spots(file_sp, summary='q05', region_labels=file_reg)
print(f"Open {file_sp}")
type(adata_pos)
adata_pos

In [ ]:
celltypes = list(adata_pos.uns['c2l_factor_names'])
lib_id    = list(adata_pos.uns['spatial'].keys())[0]

df_frac  = adata_pos.obsm['c2l_frac']

print(adata_pos.shape)
print(len(celltypes), 'cell types;', lib_id)
print(f"estimated cells / spot: median {adata_pos.obs.c2l_total_cells.median():.1f}  "
      f"(IQR {adata_pos.obs.c2l_total_cells.quantile(.25):.1f}-{adata_pos.obs.c2l_total_cells.quantile(.75):.1f})")
print(f"dominant cell-type fraction: median {df_frac.max(1).median():.2f}; spots with >= 0.5: {(df_frac.max(1) >= 0.5).sum()}")

In [ ]:
df_abund_perc = cl.build_abund_perc(adata_pos)

print(df_abund_perc.shape)
df_abund_perc

In [ ]:
df_abund = adata_pos.obsm['c2l_abund'].copy()
df_abund.head(3).T.head(3)

#### Region names

The tutorial named regions by the top-3 cell types of the max-scaled mean abundance, which favours rare types (VSMC, DC_pDC appear in many names). Here a region is named by the two cell types with the highest log2 enrichment of their **fraction** (region mean / slide mean), among cell types making up >= 2% of the region.

In [ ]:
print(df_frac.shape)
df_frac.head(3).T.head(4)

In [ ]:
regions = adata_pos.obs['region_cluster']
regions[0:5]

In [ ]:
force=False
verbose=False

df_reg, df_reg_names = cl.region_names_by_enrichment(adata_pos=adata_pos, df_frac=df_frac, regions=adata_pos.obs['region_cluster'], 
                                       n=2, min_frac=0.02,
                                       root_liana=root_liana, force=force, verbose=verbose)
print(df_reg.shape)
df_reg

In [ ]:
sel_cell_types = ['B_naive', 'B_mem', 'B_GC_LZ', 'B_GC_DZ', 'FDC', 'T_CD4+_TfH_GC', 'T_CD4+_naive', 'DC_CCR7+',
        'B_plasma', 'Endo', 'VSMC', 'Macrophages_M1']
for c in sel_cell_types:
    adata_pos.obs[f'c2l_{c}'] = df_abund[c].values          # prefixed: avoids clashes with gene names

In [ ]:
%matplotlib inline

cl.plot_abundance(adata_pos=adata_pos, sel_cell_types=sel_cell_types, 
                  title='cell2location: q05 cell abundance per spot (colour capped at the 99.2th percentile)',
                  root_fig_liana=root_fig_liana, cmap=cmap_byr, dpi=300)

### 2. Spatial weights

- Coordinates are full-resolution pixels

- the Visium centre-to-centre spacing (100 um) is 137 px. 
- A Gaussian kernel with **bandwidth=150** px (cut at weight 0.1) gives each spot: 
  - itself (1.0, **set_diag=True** - several cells share a spot), 
  - ring 1 (0.65) and 
  - ring 2 (0.18-0.28), 
  - i.e. ~18 neighbours - the same scale as the k<=20 niche of **niche_lr**.

In [ ]:
df_w = cl.calc_liana_spacing(adata_pos)
print(df_w.shape)
df_w

### 3. LIANA bivariate: spatially co-expressed LR pairs

**li.mt.bivariate** on the log-normalised spots with the **consensus** human resource:

- global: **Moran's R** (bivariate spatial autocorrelation of L and R, permutation p)

- local: **cosine** similarity of L and R within each spot's weighted neighbourhood, with **cats** (high-high, high-low, ...) and local permutation p-values

**nz_prop=0.05**: L and R (all complex subunits) expressed in >= 5% of spots.
<br>

In [ ]:
force=False
verbose=False

df_inter, adata_lr = cl.calc_liana_bivariate(root_liana=root_liana, force=force, verbose=verbose)
print(df_inter.shape)

df_inter_sig = df_inter[(df_inter.morans_fdr < 0.05) & (df_inter.morans > 0)]
print(df_inter_sig.shape)

print(f"LR pairs tested: {len(df_inter)};  Moran's R > 0 at FDR < 0.05: {len(df_inter_sig)};  R > 0.05: {(df_inter_sig.morans > 0.05).sum()}")

df_inter.head(3)


In [ ]:
cols = ['ligand', 'receptor', 'ligand_props', 'receptor_props', 'morans', 'morans_fdr']
df_inter_sig[cols].head(20)

In [ ]:
cl.plot_liana_bivariate(df_inter, adata_lr, adata_pos, 
                        N=6, root_fig_liana=root_fig_liana, 
                        cmap=cmap_byr, figsize=(16, 7.5))

### 4. LR signature per cell type

Which c2l cell types host the spatial LR patterns?

Spearman correlation across spots between the **local** LR score and each cell type's **fraction** (c2l). High df_rho = the interaction is strongest where that cell type is enriched - a host / niche, not proof that this cell type is the sender or receiver.

In [ ]:
df_frac.head(3)

In [ ]:
force=False
verbose=False

df_rho = cl.calc_liana_spearman_correlation(df_inter_sig, adata_lr, 
                                      df_frac, celltypes, root_liana, force=force, verbose=verbose)

print(df_rho.shape)
df_rho.head(3)

In [ ]:
pd.set_option('display.max_colwidth', 300)

df_rho.apply(lambda r: ', '.join(f"{c} ({r[c]:.2f})" for c in r.nlargest(8).index), axis=1).head(15).to_frame('top host cell types')

In [ ]:
cl.plot_liana_spearman_correlation(df_rho, cmap='RdBu_r', root_fig_liana=root_fig_liana)

### Claude - Opus 5.5

From the table above, can you infer biological functions and save a table at root_liana?

I'll infer a function for each of these 15 pairs from three things:
- **Program membership:** which **programs.py** programs contain the ligand or receptor.
- **Program activity at the pair's hotspots:** Spearman correlation between the pair's local score and each program's activity across spots.
- **Context from the earlier sections:** host cell types, the best cell2location sender→receiver, and the inflow source.

I'll add a short, literature-based interpretation per pair, with confidence flags where the pairing is questionable.

Program correlations align with the expected host cell types
- CXCL13/FCER2/LTB with MHC-II/humoral-B, 
- CCR7 with MHC-I/cytotoxic T-NK, and
- ACKR1/TIMP1 with chemotaxis/angiogenesis/ECM programs. 

In [ ]:
from programs.programs import *

PROGRAMS.keys()


In [ ]:
CELL_STATES.keys()

In [ ]:
print(EXCLUDE_RULES.keys())
print(REACTOME_RULES.keys())
print(CONTEXT_RULES.keys())


### summary table.

I've saved a table of inferred biological functions for the 15 pairs (15 rows × 16 columns) to **/home/flavio/uv/squidpy/results/lymph_nodes_analysis/liana/ccc_lr_biological_functions.tsv**.

Each function combines a data-driven part with a short interpretation from established lymph-node immunology:
- **Programs at the hotspots:** the 3 programs from **programs.py** (out of 51) whose activity rises most where the pair's local score is high, measured as Spearman ρ across spots.
- **Programs containing the ligand or receptor:** from **nl.annotate_lr_programs**.
- **Context from the notebook:** the top host cell types (your table), the top 3 cell2location sender→receiver pairs, and the inflow-enriched sources.
- **Interpretation:** a lymph-node compartment, a function statement, a confidence level and a caveat.

| Compartment | Pairs | Inferred function | Programs active at hotspots |
|---|---|---|---|
| T-cell zone | CCL21→CCR7, CCL19→CCR7 | Homing of naive T cells, Tregs and DCs; DC–T cell encounters (stromal CCL21; DC_CCR7+ CCL19) | MHC-I antigen presentation, cytotoxic T/NK, chemotaxis (ρ 0.27–0.35) |
| B follicle / germinal centre | FCER2→CR2, CXCL13→CXCR5 | B cell–FDC adhesion and antigen retention; CXCL13 from FDC attracting B cells and TfH into follicles and the GC light zone | MHC-II antigen presentation (0.40–0.43), humoral B cell |
| Vessels / medulla | CCL14→ACKR1, CCL2→ACKR1, CCL2→CCR2 | ACKR1 on venules presenting and clearing chemokines; recruitment of CCR2+ monocytes | Chemotaxis, angiogenesis, phagocytosis |
| Plasma-cell niche | THBS1→SDC1 | Plasma-cell adhesion and survival in the medullary cords | ER stress/UPR (0.27), as expected for secretory cells |
| Stroma / myeloid | TIMP1→CD63, THY1→ITGAX/ITGB2 | ECM remodelling and survival signalling; myeloid cells adhering to stroma | ECM remodelling, EMT, TNF/NF-κB |
| Follicle edge | CCL20→CCR6 | Positioning of CCR6+ memory and pre-GC B cells | MHC-II, ECM |

I marked 4 pairs as **low confidence**:
- **LCK→CD8A/CD8B** is an interaction inside the same cell, so its signal mainly reflects how many CD8 T cells are present.
- **LTB→CD40** is not the canonical pairing; lymphotoxin normally signals through LTBR.
- **FN1→CD79A** has weak direct evidence. Read it as fibronectin matrix next to B cells rather than a signal through the B-cell receptor.
- **VIM→CD44** involves genes expressed almost everywhere, so its local score is high in most spots.

The cell2location sender assignments follow the reference expression profiles. Because the reference has no fibroblastic reticular cells, CCL21/CCL19 are assigned to VSMC, endothelium and DC_CCR7+ instead.

In [ ]:
df_lr_bio = cl.open_lr_biological_functions(root_liana=root_liana, verbose=True)

df_lr_bio.head(3).T

### 5. LIANA inflow: source cell type & ligand -> receptor

**li.mt.inflow(obsm_key='c2l_frac')** builds, for each spot, the trivariate score **W . (ligand x proportion of source) x receptor** -> features **source^ligand^receptor**. Restricted to the 150 LR pairs with the highest global Moran's R.

Because the spot ligand is multiplied by the source **proportion**, raw inflow is dominated by abundant cell types (B_mem, B_naive, T_CD4+_naive). The table therefore also gives the source **enrichment** = log2(share of the LR total carried by the source / mean fraction of the source): > 0 when the source is over-represented around the places where L->R is high.

In [ ]:
force=False
verbose=False

df_inf = cl.save_liana_inflow(adata_pos, df_inter, df_frac,
                      root_liana=root_liana, force=force, verbose=verbose)

print(df_inf.shape)
df_inf.head(5)


In [ ]:
cl.plot_liana_inflow(df_inter, df_inf, bar_range=(-1.0, 1.0), fontsize=12, root_fig_liana=root_fig_liana)


In [ ]:
cols = ['interaction', 'source', 'global_mean', 'share', 'log2_enrichment']
df_inf[cols].head(3)

In [ ]:
lr_inf = df_inter.index[:25]

df_inf[df_inf.interaction.isin(lr_inf[:10])].groupby('interaction', sort=False).head(2)[cols].round(3)

### 6. cell2location-resolved sender -> receiver

cell2location splits every spot's counts across the 34 cell types (**layers[<cell type>]**, sum = **X**). With those layers the ligand can be taken from the **sender** cell type and the receptor from the **receiver** cell type:

**score[A,B] = mean_spots( (W_norm . L_A) x R_B )**, with **L_A**, **R_B** = log1p(CP10k) of the counts attributed to A / B (complex = min over subunits) and **W_norm** the row-normalised kernel of section 2.

Null: receptor spots permuted 200 times -> z and p (Gaussian tail of z when < 5 permutations reach the score), BH-FDR. The spot-permutation null breaks all spatial structure, so **any** co-localised pair is significant; rank pairs by **score** / **score_share** (share of the LR total) and z, not by FDR alone.

The 100 LR pairs with the highest Moran's R (FDR < 0.05) are scored (100 x 34 x 34 = 115,600 sender-receiver-LR tests).

In [ ]:
force=False
verbose=False

df_ct = cl.calc_cell2location_resolved_sender_receiver(file_sp, adata_pos, df_inter_sig, 
                                                       celltypes, root_liana, 
                                                       force=force, verbose=verbose)

df_best_ct = (df_ct[df_ct.rank_in_lr == 1].set_index('interaction')
        .reindex([i for i in df_inter_sig.index[:100] if i in set(df_ct.interaction)]))

cols = ['sender', 'receiver', 'score', 'score_share', 'z', 'fdr']
df_best_ct[cols].head(8).round(3)

In [ ]:
int(np.ceil(9/2))

In [ ]:
lista = ['CCL21^CCR7', 'CCL19^CCR7', 'CXCL13^CXCR5', 'FCER2^CR2', 'CCL2^ACKR1', 
         'FN1^CD79A', 'VIM^CD44', 'CCL14^ACKR1', 'CCL2^ACKR1', 'TIMP1^CD63', 'LTB^CD40']


cl.plot_sender_receiver_heatmap(df_ct, lista, root_fig_liana=root_fig_liana, 
                                height=5,
                                fontsize_ticks=11, fontsize_title=14, fontsize_label=12)

### strongest sender -> receiver links over all LR pairs (top 40 by score, z > 5)

In [ ]:
cl.dotplot_sender_receiver(df_ct, zmin=5, nlargest=20, root_fig_liana=root_fig_liana, 
                           figsize=(4,4), cmap='Reds',dpi=100)  # cmap='viridis'
                           

### 7. LIANA rank_aggregate on the scRNA reference + c2l co-localisation

Classical, cell-type-resolved LIANA (CellPhoneDB, Connectome, log2FC, NATMI, SingleCellSignalR, CellChat-like; **rank_aggregate**) on the reference used to build the c2l signatures (**Subset**, 34 types; up to 400 cells per type, log1p(CP10k)). It is non-spatial, so each source -> target pair is annotated with the **c2l co-localisation** of the two cell types: Pearson r between the abundance of A at a spot and the kernel-weighted abundance of B around it (symmetrised).

"Spatially supported" = **specificity_rank <= 0.05**, **magnitude_rank <= 0.05** and co-localisation > 0.2 (with 100 permutations the specificity rank of most specific pairs bottoms out near 0.03, so 0.01 would keep only a handful).

In [ ]:
df_abund.head(4).T.head(10)

In [ ]:
force=False
verbose=False

df_liref = cl.calc_liana_rank_aggregate_reference(file_ref, root_liana=root_liana, force=force, verbose=verbose)

df_liref.head(3)

In [ ]:
force=False
verbose=True

df_coloc, df_liref_sel, df_liref_sup = cl.save_celltype_colocalisation(adata_pos, df_abund, df_liref, 
                                                         root_liana=root_liana, force=force, verbose=verbose)

df_coloc.head(3)

In [ ]:
cols = ['source', 'target', 'ligand_complex', 'receptor_complex', 'magnitude_rank', 'specificity_rank', 'coloc']

print(df_liref_sup.shape)
df_liref_sup[cols].head(6).round(4)

In [ ]:
print(f"reference interactions: {len(df_liref)};  specific & strong: {len(df_liref_sel)};  + co-localised (r > 0.2): {len(df_liref_sup)}")

In [ ]:
cl.plot_cell_type_colocalisation_spot_neighborhood_heatmap(df_coloc, cmap='RdBu_r', vRange=(-1, 1), root_fig_liana=root_fig_liana)

Germinal-centre compartment (FDC, GC B, GC TfH, tingible-body-like macrophages - cell types that co-localise in section 7), spatially supported interactions only:

In [ ]:
gc_types = ['FDC', 'T_CD4+_TfH_GC', 'B_GC_LZ', 'B_GC_DZ', 'B_GC_prePB', 'B_Cycling', 'Macrophages_M1']

p = cl.plot_gc_panels(df_liref_sup, gc_types, root_fig_liana=root_fig_liana)
if p:
    p.show()

### 8. MISTy: spatial dependencies between c2l cell types

**li.mt.genericMistyData** on the c2l **fractions**: each cell type is predicted from the other cell types in its **juxta** view (6 nearest spots = ring 1) and **para** view (Gaussian, bandwidth 275 px ~ 2 rings). **bypass_intra=True**: fractions of one spot sum to 1, so the intra view would predict a target trivially.

**gain_R2** = variance explained by the spatial context; the importances show which neighbouring cell types predict each target.

In [ ]:
force=False
verbose=True

misty, df_misty_metrics, df_misty_inter = cl.calc_misty(adata_pos, df_frac, celltypes,
                                            root_liana=root_liana, force=force, verbose=verbose)

print(type(misty))
misty

In [ ]:
print(df_misty_metrics.shape)
df_misty_metrics.head(6)

In [ ]:
print(df_misty_inter.shape)
df_misty_inter.target.unique()

In [ ]:
df_misty_inter.head(6)

In [ ]:
fig = cl.plot_misty(misty, root_fig_liana, dpi=300, figure_size=(10, 8))

if fig:
    fig.show()

In [ ]:
imp = (misty.uns['interactions'].query("view == 'juxta'")
       .pivot(index='predictor', columns='target', values='importances').fillna(0))

fig = cl.plotly_clustermap(imp,
                           width=1100, height=1000, 
                           root_fig_liana=root_fig_liana)

if fig:fig.show()


### 9. Niche LR + biological programs (**niche_lr**, **programs**) on c2l regions

As in **squid03_niche_LigRec_programs.ipynb**, now with the human consensus resource and the human **PROGRAMS** / **CELL_STATES**:
- one centre per c2l region (**nl.centers_by_group**: most interior spot), here the **germinal-centre** region (highest B_GC_LZ + FDC fraction)
- centre pooled with ring 1 (**center_pool=6**), partners = remaining 14 spots; background = 2,000 niches centred **outside** the region
- both directions (center->neighbors, neighbors->center), z / FDR vs background, gradients along 3-6 collinear spots, program activity and inferred functions

New here (c2l): the niche's **cell-type composition** vs background, and the **sender / receiver cell types** of each niche LR = share of the ligand (receptor) counts attributed to each cell type in the c2l layers.

In [ ]:
df_frac.head(3)

In [ ]:
s = (df_reg_names.set_index('region_cluster')['region_name'])
s.index = s.index.astype(str)
reg_names = s.to_dict()
reg_names

In [ ]:
force=False
verbose=False

res, df_comp, df_lr, df_function, df_att, dfr, df_states, df_chk =\
    cl.calc_centers_by_group_B_GC_LZ_and_FDC(adata_pos, adata_lr, df_frac, 
                                             reg_names, file_sp, root_liana=root_liana, 
                                             force=force, verbose=verbose)

print(df_comp.shape)
df_comp.head(8).round(3)

In [ ]:
print(df_lr.shape)
df_lr.head(6)

In [ ]:
print(df_att.shape)
df_att.head(6)

In [ ]:
print(df_function.shape)
cols=['category', 'program', 'z', 'percentile', 'lr_support', 'n_lr', 'top_lr', 'score']
df_function.head(12).round(3)

In [ ]:
print(dfr.shape)
dfr.head(12).round(4)

In [ ]:
print(df_states.shape)
df_states.round(3)

In [ ]:
# LIANA cross-check: local cosine at the centre / neighbours and global rank (Moran's R) of the niche LRs
print(df_chk.shape)
df_chk.round(3)

In [ ]:
importlib.reload(cl)

plt = cl.plot_neighbor_to_center(adata_pos, res, df_frac, reg_names, root_fig_liana, dpi=300)

if plt:
    plt.show()

In [ ]:
if len(res.gradients):
    gc_score = (df_frac['B_GC_LZ'] + df_frac['FDC']).groupby(adata_pos.obs['region_cluster'].astype(str)).mean()
    region_gc = gc_score.idxmax()
    
    keys = res.gradients.sort_values(['consistent_with_direction', 'z'], ascending=False).key.head(6).tolist()
    fig = nl.plot_gradients(res, keys)
    fig.savefig(root_fig_liana / f'niche_gradients_region{region_gc}.png', dpi=300, bbox_inches='tight')
    plt.show()

In [ ]:
cols=['key', 'n_cells', 'L_rho', 'R_rho', 'pattern', 'consistent_with_direction', 'z', 'fdr']
res.gradients[cols].round(3)

#### All c2l regions: program activity per region niche

**nl.scan_niches**: one niche per region, each compared with niches centred outside its own region (1,000 background niches).

In [ ]:
import logging
logging.getLogger('choreographer').setLevel(logging.WARNING)

In [ ]:
importlib.reload(cl)

force=False
verbose=True

df_mat, niche_results = cl.calc_niche_functions_by_c2l_region(adata_pos, regions=adata_pos.obs['region_cluster'], 
                                               df_frac=df_frac, min_frac=0.02, n=3,
                                               root_liana=root_liana, force=force, verbose=verbose)

print(df_mat.shape)
df_mat.head(8).T.head(8).round(3)

In [ ]:
# niche_results

In [ ]:
class DropKaleido(logging.Filter):
    def filter(self, rec):
        return not (rec.name.startswith(('kaleido', 'choreographer'))
                    or rec.pathname.find('/choreographer/') >= 0
                    or rec.pathname.find('/kaleido/') >= 0)

for h in logging.getLogger().handlers:
    h.addFilter(DropKaleido())

In [ ]:
importlib.reload(cl)

logging.getLogger('choreographer').setLevel(logging.WARNING)

fig = cl.plotly_niche_functions_clustermap(df_mat, method='average', metric='euclidean', cmap='RdBu_r',
                                          vmax_q = 0.99, width = 1400, height = 1400, font = 14,
                                          cbar_len = 0.4, root_fig_liana=root_fig_liana)
                                          
if fig:
    fig.show()

### 10. Consensus across methods

Per LR pair (top 100 by Moran's R): global spatial co-expression (bivariate), best c2l-resolved sender -> receiver, most enriched inflow source, best spatially supported reference pair (rank_aggregate + co-localisation) and the number of region niches where the pair is significant (niche_lr FDR <= 0.1).

In [ ]:
df_inter.head(3)

In [ ]:
importlib.reload(cl)

force=True
verbose=True

df_cons = cl.calc_consensus_across_methods(df_inter_sig=df_inter_sig, df_inter=df_inter,
                                           df_best_ct=df_best_ct, df_inf=df_inf, df_liref_sup=df_liref_sup, 
                                           niche_results=niche_results,    
                                           root_liana=root_liana, force=force, verbose=verbose)

df_cons.head(5).T.round(3)

### Notes for interpretation

- **Spot level.** Bivariate / inflow / niche scores are computed on ~27-cell spots; a high score means L and R co-vary in space at ~100-200 um, which also happens when their producers merely share a compartment.
- **Attribution = reference x abundance.** c2l layers split each spot's counts in proportion to (cell abundance x reference signature). Sender/receiver calls therefore follow the reference signatures (e.g. a gene detected in the FDC reference cluster is assigned to FDC wherever FDC are present) and cannot reveal cell-type expression that is absent from the reference.
- **Missing stroma.** The reference has no FRC / fibroblast types; CCL19/CCL21/CXCL12 counts are assigned to VSMC / Endo / DC_CCR7+.
- **Nulls.** The c2l-resolved spot-permutation null and the bivariate permutation p-values ignore spatial autocorrelation, so most co-localised pairs pass; use them to rank, and treat FDR as liberal.
- **Single section.** One lymph node section from one donor; no replicate-level inference is possible.